In [ ]:
from utils.utils import load_CSIRO, CSIRO_group_k_fold, CSIRO_stratified_group_k_fold
from torchvision.transforms import v2
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from rich.console import Console
from rich.table import Table


In [ ]:
r2_coeff = {
    "Dry_Green_g": 0.1,
    "Dry_Clover_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Total_g": 0.5,
    "GDM_g": 0.2
}
res = 1024

In [ ]:
def compute_global_mean(dataset):
    global_means = {k: 0 for k in r2_coeff.keys()}
    for row in dataset.data_values:
        for target_name, _ in r2_coeff.items():
            global_means[target_name] += row[target_name]
    for k in global_means.keys():
        global_means[k] /= len(dataset.data_values)

    return global_means

In [ ]:
def compute_r2(pred_dict, target_dict, weighted_mean, df):
    total_weighted_r2 = 0
    image_ids = df["image_path"].tolist()
    image_scores = {}
    stats = {}

    for target_name, coeff in r2_coeff.items():
        flat_preds = torch.cat(pred_dict[target_name])
        flat_targets = torch.cat(target_dict[target_name])

        # R2 score
        mse = torch.sum((flat_targets - flat_preds) ** 2)
        var = torch.sum((flat_targets - weighted_mean[target_name]) ** 2)
        r2 = 1 - mse / var
        total_weighted_r2 += coeff * r2

        # Error
        errors = (flat_targets - flat_preds) ** 2
        stats[target_name] = {
            "r2": r2.item(),
            "mse": errors.mean().item(),
            "max": errors.max().item()
        }

        image_score_pair = zip(errors.tolist(), image_ids, flat_preds.tolist(), flat_targets.tolist())
        image_score_pair = sorted(image_score_pair, key=lambda x: x[0], reverse=True)
        image_scores[target_name] = image_score_pair

    return total_weighted_r2, stats, image_scores


In [ ]:
def inference(dataloader):
    all_preds = {k: [] for k in r2_coeff.keys()}
    all_targets = {k: [] for k in r2_coeff.keys()}
    patch_preds = {
        "Tile_Dry_Green_g": [],
        "Tile_Dry_Clover_g": [],
        "Tile_Dry_Dead_g": [],
        "Tile_Gate_Dry_Green_g": [],
        "Tile_Gate_Dry_Clover_g": [],
        "Tile_Gate_Dry_Dead_g": []
    }
    for data_dict in tqdm(dataloader):
        for k, v in data_dict.items():
            if isinstance(v, torch.Tensor):
                data_dict[k] = v.to(device)

        # input_imgs = data_dict["Input_Img"].view(data_dict["Input_Img"].shape[0] * 2, 3, res, res)
        # pred_dict = model(input_imgs, return_patch_preds=True, return_gates=True)
        b_tmp = data_dict["HR_Input_Img"].shape[0]
        hr_input_imgs = data_dict["HR_Input_Img"].view(b_tmp * 2, 3, res, res)
        lr_input_imgs = data_dict["LR_Input_Img"].view(b_tmp * 2, 3, res // 2, res // 2)
        pred_dict = model(hr_input_imgs, lr_input_imgs, return_patch_preds=True, return_gates=True)

        for k in r2_coeff.keys():
            all_preds[k].append(pred_dict[k].cpu())
            all_targets[k].append(data_dict[k].cpu())
            if k in ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g"]:
                patch_preds[f"Tile_{k}"].append(pred_dict[f"Tile_{k}"].cpu())
                patch_preds[f"Tile_Gate_{k}"].append(pred_dict[f"Tile_Gate_{k}"].cpu())

    return all_preds, all_targets, patch_preds

In [ ]:
from dataset import CSIROMultiScaleDataset
from model.DinoV3GatingMultiScale import DinoV3GatingMultiScale

data_folder = "../data/CSIRO"
df = load_CSIRO(data_folder)
transforms = v2.Compose([
    # v2.ToImage(),
    # v2.Resize((res, res), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
])

batch_size = 4
device = "cuda"
train_idxs, val_idxs = CSIRO_stratified_group_k_fold(df)
train_df = df.iloc[train_idxs[0]]
val_df = df.iloc[val_idxs[0]]
# train_dataset = CSIRODataset(data_folder, train_df, transforms)
# val_dataset = CSIRODataset(data_folder, val_df, transforms)
train_dataset = CSIROMultiScaleDataset(data_folder, train_df, res, res, transforms, False)
val_dataset = CSIROMultiScaleDataset(data_folder, val_df, res, res, transforms, False)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
train_global_means = compute_global_mean(train_dataset)
val_global_means = compute_global_mean(val_dataset)

# model = DinoV3Backbone(
#     model_name="facebook/dinov3-vitb16-pretrain-lvd1689m",
#     hidden_dim=128,
#     freeze_backbone=True
# )
model = DinoV3GatingMultiScale(
    model_name="facebook/dinov3-vitb16-pretrain-lvd1689m",
    hidden_dim=128,
    training_mode="full_finetune",
    predict_height=False
)
model.load_state_dict(torch.load("C:/Users/Daniel/Downloads/0_best_model_0.691.pth", weights_only=True))
model.to(device)
model.eval()

with torch.no_grad():
    train_preds, train_targets, train_patchs = inference(train_dataloader)
    val_preds, val_targets, val_patchs = inference(val_dataloader)

In [ ]:
train_r2, train_stats, train_image_scores = compute_r2(train_preds, train_targets, train_global_means, train_df)
val_r2, val_stats, val_image_scores = compute_r2(val_preds, val_targets, val_global_means, val_df)

# 2. Build the Master Table
console = Console()
table = Table(title="CSIRO Model Performance", show_header=True, header_style="bold magenta")

# Define Columns
table.add_column("Target", style="cyan", no_wrap=True)
table.add_column("Train R2", justify="right", style="green")
table.add_column("Val R2", justify="right", style="green")
table.add_column("Train MSE", justify="right", style="yellow")
table.add_column("Val MSE", justify="right", style="yellow")
# Optional: Add Max Error if you want detail
# table.add_column("Val MaxErr", justify="right", style="red")

# 3. Add Rows for each Target
for target in r2_coeff.keys():
    t_dat = train_stats[target]
    v_dat = val_stats[target]

    # Highlight overfitting (if Val R2 is significantly lower than Train R2)
    r2_color = "red" if (t_dat['r2'] - v_dat['r2']) > 0.1 else "green"

    table.add_row(
        target,
        f"{t_dat['r2']:.4f}",
        f"[{r2_color}]{v_dat['r2']:.4f}[/]",  # Conditional coloring
        f"{t_dat['mse']:.4f}",
        f"{v_dat['mse']:.4f}",
    )

# 4. Add Summary Row (Weighted R2)
table.add_section()
table.add_row(
    "[bold]Weighted Score[/]",
    f"[bold]{train_r2:.4f}[/]",
    f"[bold]{val_r2:.4f}[/]",
    "-", "-"
)

console.print(table)

In [ ]:
import matplotlib.pyplot as plt
import torch
import os
from torchvision.transforms import v2
from torchvision.io import read_image

def visualize_patch(image_path, patch_pred, gate_pred, error, pred, target, ax_img, ax_map, ax_gate, height, species, data_folder="../data/CSIRO"):
    """
    Draws the image, prediction heatmap, and gate heatmap on provided axes.
    """
    # --- 1. Image Processing ---
    img_full_path = os.path.join(data_folder, image_path)
    if not os.path.exists(img_full_path):
        print(f"Warning: Image not found at {img_full_path}")
        return

    img = read_image(img_full_path)
    _, _, w = img.shape
    center = w // 2
    left_img = v2.Resize((res, res), antialias=True)(img[:, :, :center])
    right_img = v2.Resize((res, res), antialias=True)(img[:, :, center:])
    full_img = torch.cat([left_img, right_img], dim=2).permute(1, 2, 0)

    # --- 2. Prepare Maps ---
    # Reshape logic: (2 images, 48*48 patches) -> (2, 48, 48)
    reshape_pred = patch_pred.reshape(2, int(res / 16), int(res / 16))
    pred_map = torch.cat([reshape_pred[0], reshape_pred[1]], dim=1)

    reshape_gate = gate_pred.reshape(2, int(res / 16), int(res / 16))
    gate_map = torch.cat([reshape_gate[0], reshape_gate[1]], dim=1)

    # --- 3. Plotting ---

    # Row 1: Real Image
    ax_img.imshow(full_img)
    ax_img.axis("off")
    ax_img.set_title("Input Image", fontsize=10)

    # Row 2: Prediction Heatmap
    im_pred = ax_map.imshow(pred_map.float(), cmap="jet", interpolation="nearest") # Ensure float for display
    ax_map.axis("off")
    plt.colorbar(im_pred, ax=ax_map, fraction=0.046, pad=0.04)

    # --- UPDATED TITLE LOGIC ---
    # \n creates a new line. We format float to 2 decimal places.
    stats_text = (
        f"Err: {error:.4f}\n"
        f"Pred: {pred:.2f} | Tgt: {target:.2f}\n"
        f"H: {height:.2f}cm | {species}"
    )
    ax_map.set_title(stats_text, fontsize=9, color='darkred', fontweight='bold')

    # Row 3: Gate Heatmap
    im_gate = ax_gate.imshow(gate_map.float(), cmap="magma", interpolation="nearest")
    ax_gate.axis("off")
    plt.colorbar(im_gate, ax=ax_gate, fraction=0.046, pad=0.04)
    ax_gate.set_title("Gate Attention", fontsize=10)


def visualize(patch_preds, data_df, worst_images, num_show=3):

    # 1. Prepare Data Lists from DataFrame
    image_path_list = data_df["image_path"].tolist()
    height_list = data_df["Height_Ave_cm"].tolist()
    species_list = data_df["Species"].tolist()

    # 2. Create Lookup Dictionaries
    # Maps image_path -> specific value
    image_2_height = {path: h for path, h in zip(image_path_list, height_list)}
    image_2_species = {path: s for path, s in zip(image_path_list, species_list)}

    for k in ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g"]:

        # --- Check if keys exist in patch_preds ---
        if f"Tile_{k}" not in patch_preds:
            continue

        # Flatten Prediction & Gate Patches
        flat_patch_preds = torch.cat(patch_preds[f"Tile_{k}"]).reshape(-1, 2, int(res * res / (16 * 16)), 1)
        flat_gate_preds = torch.cat(patch_preds[f"Tile_Gate_{k}"]).reshape(-1, 2, int(res * res / (16 * 16)), 1)

        # Map tensors to image paths
        image_2_patch = {path: preds for path, preds in zip(image_path_list, flat_patch_preds)}
        image_2_gate = {path: gate for path, gate in zip(image_path_list, flat_gate_preds)}

        # Get the samples to show
        samples = worst_images.get(k, [])[:num_show]

        if len(samples) == 0:
            continue

        # --- Create Figure ---
        fig, axes = plt.subplots(nrows=3, ncols=len(samples), figsize=(4 * len(samples), 10))

        # Handle case where only 1 sample exists (axes is 1D array)
        if len(samples) == 1:
            axes = axes.reshape(2, 1)

        fig.suptitle(f"Target Variable: {k} (Worst {len(samples)})", fontsize=16)

        for i, (error, img_path, pred, tgt) in enumerate(samples):

            # --- Retrieve Metadata ---
            # Retrieve height and species using the dictionaries created earlier
            cur_height = image_2_height.get(img_path, 0.0)
            cur_species = image_2_species.get(img_path, "Unknown")

            visualize_patch(
                image_path=img_path,
                patch_pred=image_2_patch[img_path],
                gate_pred=image_2_gate[img_path],
                #gate_pred=None,
                error=error,
                pred=pred,
                target=tgt,
                ax_img=axes[0, i],
                ax_map=axes[1, i],
                ax_gate=axes[2, i],
                height=cur_height,   # Pass Height
                species=cur_species  # Pass Species
            )

        plt.tight_layout()
        plt.show()

# --- Example Usage ---
# Note: Ensure the last argument is an integer (num_show), not a list like 'train_best_images'
# If you want to visualize best images, call the function a second time passing best_images as the 3rd arg.

print("Visualizing Train Worst:")
visualize(train_patchs, train_df, train_image_scores, num_show=10)

print("Visualizing Val Worst:")
visualize(val_patchs, val_df, val_image_scores, num_show=10)

In [ ]:
del model
del df
del train_dataset
del train_dataloader
del val_dataset
del val_dataloader

| Metric | Model 1 Train | Model 1 Val | Model 2 Train | Model 2 Val | Model 3 Train | Model 3 Val | Model 4 Train | Model 4 Val |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Main Loss** | 3.4208 | 4.4226 | 2.1489 | 4.1754 | 4.9879 | 7.0076 | 2.0152 | 10.3557 |
| **R2 Score (Weighted)** | 0.8459 | 0.4489 | 0.9232 | 0.7631 | 0.6854 | 0.6125 | 0.8947 | 0.5191 |
| **Dry_Green_g R2** | 0.9118 | 0.3420 | 0.9583 | 0.8377 | 0.7799 | 0.7800 | 0.9570 | 0.4910 |
| **Dry_Clover_g R2** | 0.8787 | 0.4508 | 0.9029 | 0.8883 | 0.6497 | 0.7142 | 0.9486 | 0.4539 |
| **Dry_Dead_g R2** | 0.6684 | 0.3581 | 0.8451 | 0.5658 | 0.5267 | 0.2758 | 0.7464 | -0.0070 |
| **Dry_Total_g R2** | 0.8450 | 0.4790 | 0.9274 | 0.7263 | 0.6840 | 0.5661 | 0.8801 | 0.6983 |
| **GDM_g R2** | 0.8874 | 0.4716 | 0.9442 | 0.8540 | 0.7387 | 0.7622 | 0.9472 | 0.3809 |